# 🌀 Spirale Dorée et Fractales

> *La spirale de Fibonacci et les fractales sont les signatures visuelles de l'harmonie naturelle — les mêmes structures que phi-complexity cherche dans le code.*

Ce notebook explore :
1. La spirale dorée de Fibonacci (angle 137.5°)
2. La dimension fractale de l'AST
3. L'auto-similarité dans la structure du code

**Prérequis** : `pip install phi-complexity[notebooks]`

In [ ]:
import ast
import math
import os
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

from phi_complexity.core import PHI, PHI_INV, SEQUENCE_FIBONACCI
from phi_complexity import auditer

## 1. La Spirale Dorée de Fibonacci

L'angle doré de **137.5°** produit la distribution la plus uniforme possible des points sur un disque — c'est la phyllotaxie des tournesols et des pommes de pin.

$$\theta_n = n \times 137.507764°$$
$$r_n = \sqrt{n}$$

In [ ]:
# Spirale dorée avec angle de 137.5°
angle_dore = 137.5 * (math.pi / 180)
n_points = 500

indices = np.arange(1, n_points + 1)
r = np.sqrt(indices)
theta = indices * angle_dore

x = r * np.cos(theta)
y = r * np.sin(theta)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Spirale dorée
colors = plt.cm.magma(np.linspace(0.1, 0.9, n_points))
ax1.scatter(x, y, c=colors, s=15, alpha=0.9, edgecolors="none")
ax1.set_title(f"Spirale Dorée (θ = 137.5°, n = {n_points})", fontsize=14)
ax1.set_aspect("equal")
ax1.axis("off")

# Comparaison : angle non-doré (120° = rationnel)
angle_bad = 120 * (math.pi / 180)
x_bad = r * np.cos(indices * angle_bad)
y_bad = r * np.sin(indices * angle_bad)

ax2.scatter(x_bad, y_bad, c=colors, s=15, alpha=0.9, edgecolors="none")
ax2.set_title("Angle Rationnel (θ = 120°) — Gaps visibles", fontsize=14)
ax2.set_aspect("equal")
ax2.axis("off")

plt.suptitle("Phyllotaxie : l'Angle Doré vs un Angle Rationnel", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Angle doré = 360° × (1 - 1/φ) = 360° × {1 - PHI_INV:.6f} = {360 * (1 - PHI_INV):.4f}°")

## 2. Spirales de Fibonacci Imbriquées

Les rectangles de Fibonacci forment une spirale logarithmique — la même structure que phi-complexity utilise pour évaluer la taille des fonctions.

In [ ]:
# Rectangles de Fibonacci avec spirale
fig, ax = plt.subplots(figsize=(12, 8))

fib = SEQUENCE_FIBONACCI[:9]  # 1, 1, 2, 3, 5, 8, 13, 21, 34
colors_fib = plt.cm.magma(np.linspace(0.2, 0.9, len(fib)))

# Construction des rectangles empilés
x0, y0 = 0, 0
direction = 0  # 0=droite, 1=haut, 2=gauche, 3=bas

positions = []  # Pour tracer la spirale
for i, f in enumerate(fib):
    rect = patches.Rectangle((x0, y0), f, f, linewidth=1.5,
                              edgecolor="black", facecolor=colors_fib[i], alpha=0.6)
    ax.add_patch(rect)
    ax.text(x0 + f/2, y0 + f/2, str(f), ha="center", va="center",
            fontsize=max(6, f), fontweight="bold")
    positions.append((x0, y0, f))

    # Déplacer selon la direction
    if direction == 0:  # Droite
        x0 += f
    elif direction == 1:  # Haut
        y0 += f
    elif direction == 2:  # Gauche
        x0 -= fib[i + 1] if i + 1 < len(fib) else f
    elif direction == 3:  # Bas
        y0 -= fib[i + 1] if i + 1 < len(fib) else f
    direction = (direction + 1) % 4

# Tracer la spirale approximative
t = np.linspace(0, 4 * math.pi, 500)
a = 1.0
b = math.log(PHI) / (math.pi / 2)
spiral_r = a * np.exp(b * t)
spiral_x = spiral_r * np.cos(t) + fib[0]
spiral_y = spiral_r * np.sin(t) + fib[0]
mask = spiral_r < max(fib) * 1.5
ax.plot(spiral_x[mask], spiral_y[mask], color="#FFD700", linewidth=2.5, alpha=0.8)

ax.set_xlim(-max(fib) - 5, sum(fib[:4]) + 5)
ax.set_ylim(-max(fib) - 5, sum(fib[:4]) + 5)
ax.set_aspect("equal")
ax.set_title("Rectangles de Fibonacci et Spirale Logarithmique", fontsize=14)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 3. Dimension Fractale de l'AST

L'arbre syntaxique abstrait (AST) d'un programme possède une structure fractale — la complexité se répète à chaque niveau d'imbrication.

On estime la **dimension fractale** par la méthode de box-counting sur la profondeur de l'AST.

In [ ]:
# Dimension fractale de l'AST via box-counting
def ast_depths(node, depth=0):
    """Retourne la profondeur de chaque nœud de l'AST."""
    depths = [depth]
    for child in ast.iter_child_nodes(node):
        depths.extend(ast_depths(child, depth + 1))
    return depths

def box_counting_dimension(depths):
    """Estime la dimension fractale par box-counting simplifié."""
    if not depths or max(depths) == 0:
        return 0.0
    max_depth = max(depths)
    # Compter le nombre de boîtes nécessaires à différentes échelles
    scales = []
    counts = []
    for box_size in range(1, max_depth + 1):
        n_boxes = len(set(d // box_size for d in depths))
        if n_boxes > 0:
            scales.append(box_size)
            counts.append(n_boxes)

    if len(scales) < 2:
        return 1.0

    # Régression log-log
    log_scales = [math.log(1.0 / s) for s in scales]
    log_counts = [math.log(c) for c in counts]

    # Pente par moindres carrés
    n = len(log_scales)
    sum_x = sum(log_scales)
    sum_y = sum(log_counts)
    sum_xy = sum(x * y for x, y in zip(log_scales, log_counts))
    sum_x2 = sum(x * x for x in log_scales)
    denom = n * sum_x2 - sum_x * sum_x
    if denom == 0:
        return 1.0
    slope = (n * sum_xy - sum_x * sum_y) / denom
    return abs(slope)

# Analyser des fichiers du projet
fichiers_analyse = []
for root, dirs, files in os.walk(os.path.join("..", "phi_complexity")):
    for f in files:
        if f.endswith(".py") and not f.startswith("__"):
            fichiers_analyse.append(os.path.join(root, f))

resultats_fractal = []
for fpath in fichiers_analyse[:12]:
    try:
        with open(fpath, "r", encoding="utf-8") as f:
            source = f.read()
        tree = ast.parse(source)
        depths = ast_depths(tree)
        dim = box_counting_dimension(depths)
        resultats_fractal.append({
            "fichier": os.path.basename(fpath),
            "dimension": dim,
            "profondeur_max": max(depths),
            "nb_noeuds": len(depths),
        })
    except Exception:
        pass

# Visualisation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

noms = [r["fichier"][:15] for r in resultats_fractal]
dims = [r["dimension"] for r in resultats_fractal]
profondeurs = [r["profondeur_max"] for r in resultats_fractal]

colors_dim = ["#FFD700" if d < 1.0 else "#FF6347" if d > 1.5 else "#1E90FF" for d in dims]
ax1.barh(noms, dims, color=colors_dim, edgecolor="black", linewidth=0.5)
ax1.axvline(x=1.0, color="gold", linestyle="--", linewidth=2, label="Dim = 1 (linéaire)")
ax1.set_xlabel("Dimension Fractale", fontsize=12)
ax1.set_title("Dimension Fractale de l'AST", fontsize=13)
ax1.legend()

ax2.scatter(profondeurs, dims, c=dims, cmap="magma", s=100, edgecolors="black")
ax2.set_xlabel("Profondeur Max de l'AST", fontsize=12)
ax2.set_ylabel("Dimension Fractale", fontsize=12)
ax2.set_title("Profondeur vs Dimension Fractale", fontsize=13)
ax2.grid(True, alpha=0.3)

plt.suptitle("Analyse Fractale du Code Source", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

for r in resultats_fractal:
    print(f"  {r['fichier']:20s}  dim={r['dimension']:.4f}  "
          f"prof_max={r['profondeur_max']}  noeuds={r['nb_noeuds']}")

## 4. Auto-Similarité — La Cascade Dorée

Un code bien structuré montre de l'auto-similarité à travers les niveaux d'abstraction : les patterns se répètent de la fonction individuelle au module, puis au projet entier.

C'est le principe de la **Cohérence Bayésienne Dorée** : `κ[i+1]/κ[i] → φ`

In [ ]:
# Visualiser la cascade dorée (ratios consécutifs)
try:
    metriques = auditer("../examples/code_harmonieux.py")
    # Utiliser les complexités des fonctions
    annotations = metriques.get("annotations", [])
    nb_fonctions = metriques.get("nb_fonctions", 0)

    # Séquence de Fibonacci idéale pour comparaison
    n = max(nb_fonctions, 5)
    fib_ideal = SEQUENCE_FIBONACCI[:n]
    ratios_ideal = [fib_ideal[i+1] / fib_ideal[i] for i in range(len(fib_ideal) - 1)]

    fig, ax = plt.subplots(figsize=(12, 5))
    x = range(len(ratios_ideal))
    ax.plot(x, ratios_ideal, "o-", color="#FFD700", linewidth=2, markersize=8,
            label="Ratios Fibonacci consécutifs")
    ax.axhline(y=PHI, color="#FF6347", linestyle="--", linewidth=2,
               label=f"φ = {PHI:.4f} (attracteur)")
    ax.fill_between(x, PHI - 0.1, PHI + 0.1, alpha=0.1, color="gold",
                    label="Zone dorée (±0.1)")

    ax.set_xlabel("Index de paire consécutive", fontsize=12)
    ax.set_ylabel("Ratio κ[i+1] / κ[i]", fontsize=12)
    ax.set_title("Convergence des Ratios Fibonacci vers φ\n(Auto-similarité fractale)", fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"Ratios consécutifs de Fibonacci : {[f'{r:.4f}' for r in ratios_ideal]}")
    print(f"Convergence vers φ = {PHI:.6f}")
except FileNotFoundError:
    print("Fichier exemple non trouvé — lancez depuis le dossier notebooks/")

## 5. Conclusion — Le Code comme Fractal

| Concept Fractal | Application au Code |
|---|---|
| **Auto-similarité** | Les patterns se répètent à chaque niveau d'abstraction |
| **Dimension fractale** | Mesure la complexité structurelle de l'AST |
| **Spirale dorée** | L'angle 137.5° = distribution optimale des responsabilités |
| **Cascade dorée** | κ[i+1]/κ[i] → φ = cohérence entre les niveaux |

> *"Un code fractal n'est pas répétitif — il est auto-similaire. La même harmonie résonne de la ligne au module."*

---
*Ancré dans la Bibliothèque Céleste — Morphic Phi Framework — 2026*